# Parkinson's Drawing Model Comparison

A participant-level comparison of classical and deep-learning image classifiers using the NewHandPD drawing dataset.

## Setup

In [ ]:
import hashlib
import platform
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

import pandas as pd
from PIL import Image

PROJECT_ROOT = Path.cwd()

print(f"Python: {platform.python_version()}")
print(f"Project root: {PROJECT_ROOT}")

## Download

In [ ]:
DATA_DIR = PROJECT_ROOT / "data"
NEWHANDPD_DIR = DATA_DIR / "raw" / "NewHandPD"
ARCHIVE_DIR = NEWHANDPD_DIR / "archives"
DOWNLOAD_MARKER = NEWHANDPD_DIR / ".download_complete"

ARCHIVES = {
    ("Healthy", "circle"): "https://wwwp.fc.unesp.br/~papa/pub/datasets/Handpd/NewHealthy/HealthyCircle.zip",
    ("Healthy", "meander"): "https://wwwp.fc.unesp.br/~papa/pub/datasets/Handpd/NewHealthy/HealthyMeander.zip",
    ("Healthy", "spiral"): "https://wwwp.fc.unesp.br/~papa/pub/datasets/Handpd/NewHealthy/HealthySpiral.zip",
    ("PD", "circle"): "https://wwwp.fc.unesp.br/~papa/pub/datasets/Handpd/NewPatients/PatientCircle.zip",
    ("PD", "meander"): "https://wwwp.fc.unesp.br/~papa/pub/datasets/Handpd/NewPatients/PatientMeander.zip",
    ("PD", "spiral"): "https://wwwp.fc.unesp.br/~papa/pub/datasets/Handpd/NewPatients/PatientSpiral.zip",
}

download_required = not DOWNLOAD_MARKER.exists()

if download_required:
    ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

    for (label, drawing_type), url in ARCHIVES.items():
        archive_path = ARCHIVE_DIR / f"{label.lower()}_{drawing_type}.zip"
        extraction_dir = NEWHANDPD_DIR / label / drawing_type
        extraction_dir.mkdir(parents=True, exist_ok=True)

        if not archive_path.exists():
            temporary_path = archive_path.with_suffix(".zip.part")
            print(f"Downloading {archive_path.name}...")
            urlretrieve(url, temporary_path)
            temporary_path.replace(archive_path)

        with ZipFile(archive_path) as archive:
            members = [
                member
                for member in archive.infolist()
                if "__MACOSX" not in Path(member.filename).parts
                and not Path(member.filename).name.startswith("._")
            ]
            extraction_root = extraction_dir.resolve()
            member_paths = [
                (extraction_dir / member.filename).resolve()
                for member in members
            ]
            if any(not path.is_relative_to(extraction_root) for path in member_paths):
                raise ValueError(f"Unsafe path found in {archive_path.name}")
            archive.extractall(extraction_dir, members=members)

image_suffixes = {".jpg", ".jpeg", ".png"}
image_paths = sorted(
    path
    for path in NEWHANDPD_DIR.rglob("*")
    if path.suffix.lower() in image_suffixes
    and "__MACOSX" not in path.parts
    and not path.name.startswith("._")
)

if len(image_paths) != 594:
    raise RuntimeError(f"Expected 594 NewHandPD images, found {len(image_paths):,}")

if download_required:
    DOWNLOAD_MARKER.touch()

print(f"Dataset folder: {NEWHANDPD_DIR}")
print(f"Downloaded now: {download_required}")
print(f"Images found: {len(image_paths):,}")
image_paths[:5]

## Inspect data

In [ ]:
if not NEWHANDPD_DIR.is_dir():
    raise FileNotFoundError(f"NewHandPD folder not found: {NEWHANDPD_DIR}")

records = []
invalid_image_paths = []

for image_path in image_paths:
    name_parts = image_path.stem.rsplit("-", maxsplit=1)
    if len(name_parts) != 2:
        invalid_image_paths.append(image_path)
        continue

    task, source_participant_code = name_parts
    participant_number = source_participant_code[1:]

    label = image_path.relative_to(NEWHANDPD_DIR).parts[0]

    if (
        label not in {"Healthy", "PD"}
        or not task[:1].isalpha()
        or not participant_number.isdigit()
    ):
        invalid_image_paths.append(image_path)
        continue

    records.append(
        {
            "path": image_path,
            "label": label,
            "participant_id": f"{label.lower()}-{int(participant_number):02d}",
            "source_participant_code": source_participant_code.upper(),
            "task": task.lower(),
        }
    )

manifest = (
    pd.DataFrame.from_records(records)
    .sort_values(["label", "participant_id", "task"])
    .reset_index(drop=True)
)

class_summary = (
    manifest.groupby("label")
    .agg(images=("path", "size"), participants=("participant_id", "nunique"))
    .sort_index()
)
task_summary = (
    manifest.groupby(["task", "label"])
    .size()
    .unstack(fill_value=0)
)

print(f"NewHandPD images: {len(manifest):,}")
print(f"Invalid image filenames: {len(invalid_image_paths):,}")
display(class_summary)
display(task_summary)
manifest.head()

## Validate images

In [ ]:
image_metadata_records = []
invalid_image_records = []

for row in manifest.itertuples(index=False):
    try:
        with Image.open(row.path) as image:
            image_format = image.format
            image_mode = image.mode
            width, height = image.size
            image.verify()

        with row.path.open("rb") as image_file:
            sha256 = hashlib.file_digest(image_file, "sha256").hexdigest()

        image_metadata_records.append(
            {
                "path": row.path,
                "format": image_format,
                "mode": image_mode,
                "width": width,
                "height": height,
                "sha256": sha256,
            }
        )
    except (OSError, SyntaxError) as error:
        invalid_image_records.append(
            {"path": row.path, "error": str(error)}
        )

invalid_images = pd.DataFrame(
    invalid_image_records,
    columns=["path", "error"],
)

if not invalid_images.empty:
    display(invalid_images)
    raise RuntimeError(f"Image validation failed for {len(invalid_images)} files")

image_metadata = pd.DataFrame.from_records(image_metadata_records)
validated_manifest = manifest.merge(
    image_metadata,
    on="path",
    how="left",
    validate="one_to_one",
)

image_profiles = (
    validated_manifest.groupby(["format", "mode", "width", "height"])
    .size()
    .rename("images")
    .reset_index()
    .sort_values("images", ascending=False)
)

duplicate_images = validated_manifest[
    validated_manifest.duplicated("sha256", keep=False)
].sort_values(["sha256", "label", "participant_id", "task"])
duplicate_summary = (
    duplicate_images.groupby("sha256")
    .agg(
        files=("path", "size"),
        participants=("participant_id", "nunique"),
        labels=("label", "nunique"),
        tasks=("task", "nunique"),
    )
    .reset_index()
)

expected_tasks = {"circa", "mea1", "mea2", "mea3", "mea4", "sp1", "sp2", "sp3", "sp4"}
participant_task_sets = validated_manifest.groupby("participant_id")["task"].agg(set)
task_issue_records = []

for participant_id, observed_tasks in participant_task_sets.items():
    missing_tasks = sorted(expected_tasks - observed_tasks)
    unexpected_tasks = sorted(observed_tasks - expected_tasks)
    if missing_tasks or unexpected_tasks:
        task_issue_records.append(
            {
                "participant_id": participant_id,
                "missing_tasks": missing_tasks,
                "unexpected_tasks": unexpected_tasks,
            }
        )

task_issues = pd.DataFrame(
    task_issue_records,
    columns=["participant_id", "missing_tasks", "unexpected_tasks"],
)

expected_prefix = {"Healthy": "H", "PD": "P"}
participant_code_issues = validated_manifest[
    validated_manifest.apply(
        lambda row: row.source_participant_code[:1] != expected_prefix[row.label],
        axis=1,
    )
]["path label participant_id source_participant_code task".split()]

print(f"Images validated: {len(validated_manifest):,}")
print(f"Unreadable images: {len(invalid_images):,}")
print(f"Exact duplicate groups: {len(duplicate_summary):,}")
print(f"Duplicate groups spanning participants: {(duplicate_summary['participants'] > 1).sum():,}")
print(f"Duplicate groups spanning labels: {(duplicate_summary['labels'] > 1).sum():,}")
print(f"Participants with task issues: {len(task_issues):,}")
print(f"Participant-code prefix issues: {len(participant_code_issues):,}")
display(image_profiles.head(10))
display(task_issues)
participant_code_issues.head()